# AraContract Analyzer - Model Training
## Google Colab Notebook with Google Drive Checkpoints

This notebook trains the AraContract clause classification model on Google Colab with GPU acceleration and saves checkpoints directly to Google Drive.

**Model:** CAMeLBERT (Arabic BERT) for multi-task classification (clause type + risk level)

**Dataset:** AraContract JSONL format with `text`, `type_clause`, and `risk_level` fields

## 1. Setup and Configuration

In [1]:
#@title Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print("✓ Google Drive mounted at /content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Google Drive mounted at /content/drive


In [2]:
#@title Check GPU Availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
CUDA version: 12.8


In [3]:
#@title Install Required Packages
# We use numpy<2.0 to maintain compatibility with scikit-learn 1.4.0
!pip install -q transformers==4.37.0 scikit-learn==1.4.0 'numpy<2.0'
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121
print("✓ Packages installed. IMPORTANT: Please go to Runtime -> Restart session now.")

✓ Packages installed. IMPORTANT: Please go to Runtime -> Restart session now.


## 2. Configuration

In [ ]:
#@title Training Configuration
from dataclasses import dataclass
from pathlib import Path

# Google Drive folder for checkpoints (CHANGE THIS to your preferred path)
DRIVE_CHECKPOINT_FOLDER = "/content/drive/MyDrive/AraContract/checkpoints"

@dataclass
class Config:
    # Paths
    drive_checkpoint_folder: str = DRIVE_CHECKPOINT_FOLDER

    # Model
    model_name: str = "CAMeL-Lab/bert-base-arabic-camelbert-msa"
    max_seq_length: int = 512

    # Training hyperparameters
    batch_size: int = 16  # Reduce if OOM
    learning_rate: float = 2e-5
    epochs: int = 5
    warmup_ratio: float = 0.1
    weight_decay: float = 0.01
    dropout: float = 0.1
    seed: int = 42

    # Training settings
    log_interval: int = 50
    eval_interval: int = 1

    # Label mappings - 7 type classes
    type_labels: list = None
    risk_labels: list = None

    def __post_init__(self):
        self.type_labels = [
            "general_provisions",
            "payment_financial",
            "party_obligations_a",
            "party_obligations_b",
            "duration_expiration",
            "termination",
            "penalties_damages",
            "dispute_resolution",
        ]
        self.risk_labels = ["low", "medium", "high"]
        self.num_type_classes = len(self.type_labels)
        self.num_risk_classes = len(self.risk_labels)
        self.type_label_to_idx = {label: i for i, label in enumerate(self.type_labels)}
        self.risk_label_to_idx = {label: i for i, label in enumerate(self.risk_labels)}

config = Config()

# Create checkpoint directory on Drive
Path(config.drive_checkpoint_folder).mkdir(parents=True, exist_ok=True)
print(f"✓ Checkpoint folder: {config.drive_checkpoint_folder}")

✓ Checkpoint folder: /content/drive/MyDrive/AraContract/checkpoints


## 3. Dataset Loading and Preprocessing

In [5]:
#@title Dataset Statistics Check
import json
from collections import Counter

def load_jsonl(path):
    records = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

# Load data files
train_data = load_jsonl('/content/drive/MyDrive/AraContract/data/aracontract_train.jsonl')
val_data = load_jsonl('/content/drive/MyDrive/AraContract/data/aracontract_val.jsonl')
test_data = load_jsonl('/content/drive/MyDrive/AraContract/data/aracontract_test.jsonl')

print(f"Train samples: {len(train_data)}")
print(f"Validation samples: {len(val_data)}")
print(f"Test samples: {len(test_data)}")
print(f"Total: {len(train_data) + len(val_data) + len(test_data)}")

Train samples: 3609
Validation samples: 773
Test samples: 869
Total: 5251


In [6]:
#@title Check Label Distribution
print("\n=== Type Clause Distribution (Train) ===")
type_counts = Counter(r['clause_type'] for r in train_data)
for label, count in sorted(type_counts.items()):
    print(f"  {label}: {count} ({100*count/len(train_data):.1f}%)")

print("\n=== Risk Level Distribution (Train) ===")
risk_counts = Counter(r['risk_level'] for r in train_data)
for label, count in sorted(risk_counts.items()):
    print(f"  {label}: {count} ({100*count/len(train_data):.1f}%)")


=== Type Clause Distribution (Train) ===
  dispute_resolution: 357 (9.9%)
  duration_expiration: 504 (14.0%)
  general_provisions: 650 (18.0%)
  party_obligations_a: 271 (7.5%)
  party_obligations_b: 216 (6.0%)
  payment_financial: 788 (21.8%)
  penalties_damages: 523 (14.5%)
  termination: 300 (8.3%)

=== Risk Level Distribution (Train) ===
  high: 781 (21.6%)
  low: 2598 (72.0%)
  medium: 230 (6.4%)


In [7]:
#@title Sample Data Check
print("Sample training record:")
print(json.dumps(train_data[0], ensure_ascii=False, indent=2))

Sample training record:
{
  "text": "TO THE MAXIMUM EXTENT PERMITTED BY LAW, IN NO EVENT SHALL EITHER PARTY BE LIABLE OR OBLIGATED IN ANY MANNER FOR ANY SPECIAL, INCIDENTAL, EXEMPLARY OR CONSEQUENTIAL DAMAGES OF ANY KIND ARISING OUT OF OR RELATING TO THIS AGREEMENT (INCLUDING, BUT NOT LIMITED TO, LOST PROFITS, REVENUES OR BUSINESS OPPORTUNITIES) HOWEVER CAUSED AND REGARDLESS OF THE FORM OF ACTION, WHETHER IN CONTRACT, TORT, NEGLIGENCE, STRICT PRODUCT LIABILITY, OR OTHERWISE, EVEN IF THE PARTY HAS BEEN INFORMED OF THE POSSIBILITY OF ANY SUCH DAMAGES IN ADVANCE.",
  "clause_type": "penalties_damages",
  "risk_level": "high",
  "risk_reason": "Penalty or compensation - high risk",
  "language": "en",
  "contract_id": "VERTICALNETINC_04_01_2002-EX-10.19-MAINTENANCE AND SUPPORT AGREEMENT.pdf",
  "clause_position": 6,
  "total_clauses": 14,
  "parent_article_num": null,
  "sub_clause": ""
}


## 4. PyTorch Dataset and Model

In [8]:
#@title ClauseDataset Implementation
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer
from warnings import warn

class ClauseDataset(Dataset):
    def __init__(self, records, tokenizer, max_length=512, type_label_map=None, risk_label_map=None):
        self.records = records
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.type_label_map = type_label_map or config.type_label_to_idx
        self.risk_label_map = risk_label_map or config.risk_label_to_idx

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        record = self.records[idx]
        text = record.get('text', '')
        type_label = record.get('clause_type', '')
        risk_label = record.get('risk_level', '')

        # Tokenize
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        # Encode labels (default to 0 if unknown)
        type_idx = self.type_label_map.get(type_label, 0)
        risk_idx = self.risk_label_map.get(risk_label, 0)

        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'type_label': torch.tensor(type_idx, dtype=torch.long),
            'risk_label': torch.tensor(risk_idx, dtype=torch.long)
        }

print("✓ ClauseDataset defined")

✓ ClauseDataset defined


In [ ]:
#@title AraContractClassifier Model
import torch.nn as nn
from transformers import AutoModel
from sklearn.metrics import f1_score, accuracy_score

class AraContractClassifier(nn.Module):
    def __init__(self, model_name, num_type_classes=7, num_risk_classes=3, dropout_prob=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size

        self.dropout = nn.Dropout(dropout_prob)
        self.type_classifier = nn.Linear(hidden_size, num_type_classes)
        self.risk_classifier = nn.Linear(hidden_size, num_risk_classes)
        self.loss_fn = nn.CrossEntropyLoss()

    def forward(self, input_ids, attention_mask=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)

        if hasattr(outputs, 'pooler_output') and outputs.pooler_output is not None:
            pooled_output = outputs.pooler_output
        else:
            pooled_output = outputs.last_hidden_state[:, 0, :]

        pooled_output = self.dropout(pooled_output)

        type_logits = self.type_classifier(pooled_output)
        risk_logits = self.risk_classifier(pooled_output)

        return type_logits, risk_logits

    def training_step(self, input_ids, attention_mask, type_labels, risk_labels):
        type_logits, risk_logits = self(input_ids, attention_mask)
        type_loss = self.loss_fn(type_logits, type_labels)
        risk_loss = self.loss_fn(risk_logits, risk_labels)
        total_loss = type_loss + risk_loss

        loss_dict = {
            'total_loss': total_loss.item(),
            'type_loss': type_loss.item(),
            'risk_loss': risk_loss.item()
        }
        return total_loss, loss_dict

    def evaluate(self, dataloader, device):
        self.eval()
        all_type_preds, all_type_labels = [], []
        all_risk_preds, all_risk_labels = [], []

        with torch.no_grad():
            for batch in dataloader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                type_labels = batch['type_label'].to(device)
                risk_labels = batch['risk_label'].to(device)

                type_logits, risk_logits = self(input_ids, attention_mask)
                type_preds = torch.argmax(type_logits, dim=-1)
                risk_preds = torch.argmax(risk_logits, dim=-1)

                all_type_preds.extend(type_preds.cpu().numpy())
                all_type_labels.extend(type_labels.cpu().numpy())
                all_risk_preds.extend(risk_preds.cpu().numpy())
                all_risk_labels.extend(risk_labels.cpu().numpy())

        type_f1 = f1_score(all_type_labels, all_type_preds, average='weighted', zero_division=0)
        risk_f1 = f1_score(all_risk_labels, all_risk_preds, average='weighted', zero_division=0)
        type_acc = accuracy_score(all_type_labels, all_type_preds)
        risk_acc = accuracy_score(all_risk_labels, all_risk_preds)

        return {
            'type_f1': type_f1,
            'risk_f1': risk_f1,
            'type_accuracy': type_acc,
            'risk_accuracy': risk_acc
        }

print("✓ AraContractClassifier defined")

✓ AraContractClassifier defined


## 5. Training Functions

In [18]:
# @title Training Utilities
import random
import numpy as np
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, get_linear_schedule_with_warmup


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_device():
    if torch.cuda.is_available():
        return torch.device('cuda')
    return torch.device('cpu')


def build_dataloaders(train_records, val_records, tokenizer, batch_size=16, max_length=512):
    train_dataset = ClauseDataset(train_records, tokenizer, max_length)
    val_dataset = ClauseDataset(val_records, tokenizer, max_length)

    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, drop_last=False)
    val_loader = DataLoader(val_dataset, batch_size=batch_size,
                            shuffle=False, num_workers=0, drop_last=False)

    return train_loader, val_loader


def train_epoch(model, dataloader, optimizer, scheduler, device, scaler=None, log_interval=50):
    model.train()
    total_loss = 0.0
    total_type_loss = 0.0
    total_risk_loss = 0.0
    steps = 0

    for step, batch in enumerate(dataloader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        type_labels = batch['type_label'].to(device)
        risk_labels = batch['risk_label'].to(device)

        optimizer.zero_grad()

        if scaler is not None:
            with torch.cuda.amp.autocast():
                loss, loss_dict = model.training_step(
                    input_ids, attention_mask, type_labels, risk_labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss, loss_dict = model.training_step(
                input_ids, attention_mask, type_labels, risk_labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        scheduler.step()

        total_loss += loss_dict['total_loss']
        total_type_loss += loss_dict['type_loss']
        total_risk_loss += loss_dict['risk_loss']
        steps += 1

        if (step + 1) % log_interval == 0:
            print(
                f"  Step {step + 1}/{len(dataloader)} | Loss: {total_loss / steps:.4f}")

    return {
        'loss': total_loss / steps,
        'type_loss': total_type_loss / steps,
        'risk_loss': total_risk_loss / steps
    }


def save_checkpoint(model, path, run_name=None):
    from pathlib import Path
    Path(path).parent.mkdir(parents=True, exist_ok=True)

    checkpoint = {
        'model_state_dict': model.state_dict(),
        'config': {
            'model_name': config.model_name,
            'num_type_classes': config.num_type_classes,
            'num_risk_classes': config.num_risk_classes,
        }
    }
    if run_name:
        checkpoint['run_name'] = run_name

    torch.save(checkpoint, path)
    print(f"  Checkpoint saved to {path}")


print("✓ Training utilities defined")

✓ Training utilities defined


## 6. Main Training Loop

In [19]:
#@title Run Training
from datetime import datetime
import json

def train(
    epochs=5,
    batch_size=16,
    learning_rate=2e-5,
    run_name=None
):
    # Setup
    set_seed(config.seed)
    device = get_device()
    print(f"Using device: {device}")

    # Generate run name if not provided
    if run_name is None:
        run_name = f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

    # Tokenizer
    print(f"Loading tokenizer: {config.model_name}")
    tokenizer = AutoTokenizer.from_pretrained(config.model_name)

    # DataLoaders
    print("Building dataloaders...")
    train_loader, val_loader = build_dataloaders(
        train_data, val_data, tokenizer, batch_size, config.max_seq_length
    )
    print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

    # Model
    print(f"Loading model: {config.model_name}")
    model = AraContractClassifier(
        model_name=config.model_name,
        num_type_classes=config.num_type_classes,
        num_risk_classes=config.num_risk_classes,
        dropout_prob=config.dropout
    ).to(device)

    # Optimizer & scheduler
    total_steps = len(train_loader) * epochs
    warmup_steps = int(total_steps * config.warmup_ratio)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=config.weight_decay
    )
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
    )

    # Mixed precision scaler
    scaler = torch.cuda.amp.GradScaler() if device.type == 'cuda' else None

    # Training loop
    best_f1 = 0.0
    history = {'train': [], 'val': []}

    print(f"\nStarting training for {epochs} epochs...")
    print(f"Run name: {run_name}")
    print(f"Batch size: {batch_size}, LR: {learning_rate}")

    for epoch in range(1, epochs + 1):
        print(f"\n{'='*50}")
        print(f"Epoch {epoch}/{epochs}")
        print(f"{'='*50}")

        # Train
        train_metrics = train_epoch(
            model, train_loader, optimizer, scheduler, device, scaler, config.log_interval
        )
        print(f"Train loss: {train_metrics['loss']:.4f} | type: {train_metrics['type_loss']:.4f} | risk: {train_metrics['risk_loss']:.4f}")

        # Validate
        val_metrics = model.evaluate(val_loader, device)
        print(f"Val type_f1: {val_metrics['type_f1']:.4f} | risk_f1: {val_metrics['risk_f1']:.4f}")
        print(f"Val type_acc: {val_metrics['type_accuracy']:.4f} | risk_acc: {val_metrics['risk_accuracy']:.4f}")

        history['train'].append(train_metrics)
        history['val'].append(val_metrics)

        # Save best model
        avg_f1 = (val_metrics['type_f1'] + val_metrics['risk_f1']) / 2
        if avg_f1 > best_f1:
            best_f1 = avg_f1
            best_model_path = f"{config.drive_checkpoint_folder}/{run_name}_best.pt"
            save_checkpoint(model, best_model_path, run_name)
            print(f"  ★ New best model! Avg F1: {best_f1:.4f}")

        # Save epoch checkpoint
        epoch_path = f"{config.drive_checkpoint_folder}/{run_name}_epoch{epoch}.pt"
        save_checkpoint(model, epoch_path, run_name)

    # Save final model
    final_path = f"{config.drive_checkpoint_folder}/{run_name}_final.pt"
    save_checkpoint(model, final_path, run_name)

    # Save training history
    history_path = f"{config.drive_checkpoint_folder}/{run_name}_history.json"
    with open(history_path, 'w') as f:
        json.dump(history, f, indent=2)

    # Save tokenizer
    tokenizer_path = f"{config.drive_checkpoint_folder}/{run_name}_tokenizer"
    tokenizer.save_pretrained(tokenizer_path)

    print(f"\n{'='*50}")
    print(f"Training complete!")
    print(f"Best weighted F1: {best_f1:.4f}")
    print(f"Checkpoints saved to: {config.drive_checkpoint_folder}")
    print(f"{'='*50}")

    return model, tokenizer, history

print("✓ Training function defined")

✓ Training function defined


In [20]:
#@title Start Training
# Adjust these parameters as needed
EPOCHS = 5  # @param {type: "integer"}
BATCH_SIZE = 16  # @param {type: "integer"}
LEARNING_RATE = 2e-5  # @param {type: "number"}
RUN_NAME = "aracontract_v2"  # @param {type: "string"}

model, tokenizer, history = train(
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    run_name=RUN_NAME
)

Using device: cuda
Loading tokenizer: CAMeL-Lab/bert-base-arabic-camelbert-msa


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Building dataloaders...
Train batches: 226, Val batches: 49
Loading model: CAMeL-Lab/bert-base-arabic-camelbert-msa


/tmp/ipykernel_2808/1965568384.py:54: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() if device.type == 'cuda' else None
/tmp/ipykernel_2808/3671922366.py:50: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():



Starting training for 5 epochs...
Run name: aracontract_v2
Batch size: 16, LR: 2e-05

Epoch 1/5
  Step 50/226 | Loss: 2.9940
  Step 100/226 | Loss: 2.7236
  Step 150/226 | Loss: 2.4958
  Step 200/226 | Loss: 2.2981
Train loss: 2.2210 | type: 1.4531 | risk: 0.7679
Val type_f1: 0.7220 | risk_f1: 0.7885
Val type_acc: 0.7219 | risk_acc: 0.7542
  Checkpoint saved to /content/drive/MyDrive/AraContract/checkpoints/aracontract_v2_best.pt
  ★ New best model! Avg F1: 0.7553
  Checkpoint saved to /content/drive/MyDrive/AraContract/checkpoints/aracontract_v2_epoch1.pt

Epoch 2/5


...

  Step 50/226 | Loss: 0.8347
  Step 100/226 | Loss: 0.8492
  Step 150/226 | Loss: 0.8525
  Step 200/226 | Loss: 0.8297
Train loss: 0.8294 | type: 0.4768 | risk: 0.3526
Val type_f1: 0.8230 | risk_f1: 0.8834
Val type_acc: 0.8189 | risk_acc: 0.8797
  Checkpoint saved to /content/drive/MyDrive/AraContract/checkpoints/aracontract_v2_best.pt
  ★ New best model! Avg F1: 0.8532
  Checkpoint saved to /content/drive/MyDrive/AraContract/checkpoints/aracontract_v2_epoch3.pt

Epoch 4/5


/tmp/ipykernel_2808/3671922366.py:50: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


  Step 50/226 | Loss: 0.6083
  Step 100/226 | Loss: 0.6281
  Step 150/226 | Loss: 0.6368
  Step 200/226 | Loss: 0.6154
Train loss: 0.6219 | type: 0.3467 | risk: 0.2752
Val type_f1: 0.8556 | risk_f1: 0.9016
Val type_acc: 0.8525 | risk_acc: 0.8991
  Checkpoint saved to /content/drive/MyDrive/AraContract/checkpoints/aracontract_v2_best.pt
  ★ New best model! Avg F1: 0.8786
  Checkpoint saved to /content/drive/MyDrive/AraContract/checkpoints/aracontract_v2_epoch4.pt

Epoch 5/5


/tmp/ipykernel_2808/3671922366.py:50: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


  Step 50/226 | Loss: 0.5186
  Step 100/226 | Loss: 0.5035
  Step 150/226 | Loss: 0.4951
  Step 200/226 | Loss: 0.4962
Train loss: 0.4955 | type: 0.2793 | risk: 0.2162
Val type_f1: 0.8551 | risk_f1: 0.9059
Val type_acc: 0.8525 | risk_acc: 0.9030
  Checkpoint saved to /content/drive/MyDrive/AraContract/checkpoints/aracontract_v2_best.pt
  ★ New best model! Avg F1: 0.8805
  Checkpoint saved to /content/drive/MyDrive/AraContract/checkpoints/aracontract_v2_epoch5.pt
  Checkpoint saved to /content/drive/MyDrive/AraContract/checkpoints/aracontract_v2_final.pt

Training complete!
Best weighted F1: 0.8805
Checkpoints saved to: /content/drive/MyDrive/AraContract/checkpoints


## 7. Evaluation and Testing

In [21]:
# Load best model and evaluate on test set
# Define device globally if not already available
device = get_device()

# Update this path to your best model checkpoint
BEST_MODEL_PATH = f"{config.drive_checkpoint_folder}/{RUN_NAME}_best.pt"  # @param {type: "string"}

# Load checkpoint (weights_only=True for security)
checkpoint = torch.load(BEST_MODEL_PATH, map_location=device, weights_only=True)
print(f"Loaded checkpoint: {BEST_MODEL_PATH}")
print(f"Run name: {checkpoint.get('run_name', 'N/A')}")

# Reconstruct model
test_model = AraContractClassifier(
    model_name=config.model_name,
    num_type_classes=config.num_type_classes,
    num_risk_classes=config.num_risk_classes
).to(device)
test_model.load_state_dict(checkpoint['model_state_dict'])
print("✓ Model loaded")

# Create test dataloader
test_dataset = ClauseDataset(test_data, tokenizer, config.max_seq_length)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# Evaluate
print("\nEvaluating on test set...")
test_metrics = test_model.evaluate(test_loader, device)
print(f"Test type_f1: {test_metrics['type_f1']:.4f} | risk_f1: {test_metrics['risk_f1']:.4f}")
print(f"Test type_acc: {test_metrics['type_accuracy']:.4f} | risk_acc: {test_metrics['risk_accuracy']:.4f}")

if test_metrics['type_f1'] >= 0.80 and test_metrics['risk_f1'] >= 0.80:
    print("✓ Meets SRS target")
else:
    print(f"⚠ type_f1={test_metrics['type_f1']:.4f}, risk_f1={test_metrics['risk_f1']:.4f}")

Loaded checkpoint: /content/drive/MyDrive/AraContract/checkpoints/aracontract_v2_best.pt
Run name: aracontract_v2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


✓ Model loaded

Evaluating on test set...
Test type_f1: 0.8732 | risk_f1: 0.8617
Test type_acc: 0.8711 | risk_acc: 0.8585
✓ Meets SRS target


In [22]:
#@title Detailed Classification Report
from sklearn.metrics import classification_report

def get_detailed_report(model, dataloader, device, label_names):
    model.eval()
    all_type_preds, all_type_labels = [], []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            type_labels = batch['type_label'].to(device)

            type_logits, _ = model(input_ids, attention_mask)
            type_preds = torch.argmax(type_logits, dim=-1)

            all_type_preds.extend(type_preds.cpu().numpy())
            all_type_labels.extend(type_labels.cpu().numpy())

    print("\n=== Type Clause Classification Report ===")
    print(classification_report(all_type_labels, all_type_preds, target_names=label_names, digits=4))

get_detailed_report(test_model, test_loader, device, config.type_labels)


=== Type Clause Classification Report ===
                     precision    recall  f1-score   support

 general_provisions     0.9440    0.9077    0.9255       130
  payment_financial     0.9149    0.8515    0.8821       202
party_obligations_a     0.6265    0.8387    0.7172        62
party_obligations_b     0.7778    0.7568    0.7671        37
duration_expiration     0.8672    0.8538    0.8605       130
        termination     0.8061    0.8316    0.8187        95
  penalties_damages     0.9280    0.8992    0.9134       129
 dispute_resolution     0.9419    0.9643    0.9529        84

           accuracy                         0.8711       869
          macro avg     0.8508    0.8629    0.8547       869
       weighted avg     0.8784    0.8711    0.8732       869



## 8. Export Model for Deployment

In [23]:
#@title Export Model and Tokenizer for Backend
from pathlib import Path
import os

EXPORT_DIR = f"{config.drive_checkpoint_folder}/{RUN_NAME}_export"
Path(EXPORT_DIR).mkdir(parents=True, exist_ok=True)

# Save model weights only (smaller file for deployment)
export_path = f"{EXPORT_DIR}/aracontract_classifier.pt"
torch.save(test_model.state_dict(), export_path)
print(f"✓ Model weights saved: {export_path}")

# Save tokenizer
tokenizer.save_pretrained(EXPORT_DIR)
print(f"✓ Tokenizer saved: {EXPORT_DIR}")

# Save config info
export_config = {
    'model_name': config.model_name,
    'num_type_classes': config.num_type_classes,
    'num_risk_classes': config.num_risk_classes,
    'type_labels': config.type_labels,
    'risk_labels': config.risk_labels,
    'type_label_to_idx': config.type_label_to_idx,
    'risk_label_to_idx': config.risk_label_to_idx,
    'test_metrics': test_metrics,
}
config_path = f"{EXPORT_DIR}/config.json"
with open(config_path, 'w', encoding='utf-8') as f:
    json.dump(export_config, f, indent=2, ensure_ascii=False)
print(f"✓ Config saved: {config_path}")

# Print export summary
print(f"\n{'='*50}")
print("Export Summary:")
print(f"{'='*50}")
for f in os.listdir(EXPORT_DIR):
    fpath = f"{EXPORT_DIR}/{f}"
    size = os.path.getsize(fpath)
    print(f"  {f}: {size / 1024 / 1024:.1f} MB")

✓ Model weights saved: /content/drive/MyDrive/AraContract/checkpoints/aracontract_v2_export/aracontract_classifier.pt
✓ Tokenizer saved: /content/drive/MyDrive/AraContract/checkpoints/aracontract_v2_export
✓ Config saved: /content/drive/MyDrive/AraContract/checkpoints/aracontract_v2_export/config.json

Export Summary:
  aracontract_classifier.pt: 416.2 MB
  tokenizer_config.json: 0.0 MB
  special_tokens_map.json: 0.0 MB
  vocab.txt: 0.3 MB
  tokenizer.json: 0.7 MB
  config.json: 0.0 MB


## 9. Inference Test

In [24]:
#@title Test Inference on Sample Text
import torch.nn.functional as F

def predict_clause(text):
    """Predict type and risk for a single clause text."""
    test_model.eval()

    # Tokenize
    inputs = tokenizer(
        text,
        add_special_tokens=True,
        max_length=config.max_seq_length,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    ).to(device)

    # Forward pass
    with torch.no_grad():
        type_logits, risk_logits = test_model(
            inputs['input_ids'],
            inputs['attention_mask']
        )

    # Get predictions
    type_probs = F.softmax(type_logits, dim=-1).cpu().numpy()[0]
    risk_probs = F.softmax(risk_logits, dim=-1).cpu().numpy()[0]

    type_idx = type_probs.argmax()
    risk_idx = risk_probs.argmax()

    return {
        'type': config.type_labels[type_idx],
        'type_confidence': float(type_probs[type_idx]),
        'risk': config.risk_labels[risk_idx],
        'risk_confidence': float(risk_probs[risk_idx]),
        'type_probabilities': dict(zip(config.type_labels, type_probs)),
        'risk_probabilities': dict(zip(config.risk_labels, risk_probs))
    }

# Test with sample Arabic contract clauses
test_clauses = [
    "يلتزم الطرف الثاني بدفع المبلغ المتفق عليه خلال مدة لا تتجاوز 30 يوماً من تاريخ الفاتورة",
    "يحق للطرف الأول إنهاء العقد في أي وقت دون إشعار مسبق في حال إخلال الطرف الثاني بالتزاماته",
    "تبلغ مدة هذا العقد سنة واحدة قابلة للتجديد تلقائياً ما لم يخطره أحد الطرفين قبل 30 يوماً",
]

for clause in test_clauses:
    result = predict_clause(clause)
    print(f"\nClause: {clause[:50]}...")
    print(f"  Type: {result['type']} ({result['type_confidence']:.2%})")
    print(f"  Risk: {result['risk']} ({result['risk_confidence']:.2%})")


Clause: يلتزم الطرف الثاني بدفع المبلغ المتفق عليه خلال مد...
  Type: payment_financial (73.68%)
  Risk: low (93.44%)

Clause: يحق للطرف الأول إنهاء العقد في أي وقت دون إشعار مس...
  Type: party_obligations_b (21.44%)
  Risk: low (88.66%)

Clause: تبلغ مدة هذا العقد سنة واحدة قابلة للتجديد تلقائيا...
  Type: duration_expiration (70.39%)
  Risk: low (96.00%)


## 10. Download Checkpoints Locally (Optional)

In [17]:
#@title Download Best Model and Export Folder
from google.colab import files
import os

# List available checkpoints
checkpoint_files = os.listdir(config.drive_checkpoint_folder)
print("Available files in checkpoint folder:")
for f in sorted(checkpoint_files):
    fpath = f"{config.drive_checkpoint_folder}/{f}"
    if os.path.isfile(fpath):
        size = os.path.getsize(fpath)
        print(f"  {f}: {size / 1024 / 1024:.1f} MB")

# Download best model
print("\nDownloading best model...")
files.download(f"{config.drive_checkpoint_folder}/{RUN_NAME}_best.pt")

# Download config
print("Downloading config...")
files.download(f"{config.drive_checkpoint_folder}/{RUN_NAME}_history.json")

Available files in checkpoint folder:
  aracontract_v1_best.pt: 416.2 MB
  aracontract_v1_epoch1.pt: 416.2 MB
  aracontract_v1_epoch2.pt: 416.2 MB
  aracontract_v1_epoch3.pt: 416.2 MB
  aracontract_v1_epoch4.pt: 416.2 MB
  aracontract_v1_epoch5.pt: 416.2 MB
  aracontract_v1_final.pt: 416.2 MB
  aracontract_v1_history.json: 0.0 MB



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>